# Add certifier dropdowns to a procurement spreadsheet

**Web-service copy.** The food-impact service (stars.html and aia.html) runs this notebook with papermill from its processing folder: it reads `input.csv`, takes `PREP_TYPE` from the service (the cell tagged `parameters`), and writes `output/worksheet_dropdowns.xlsx`, which is emailed back. The dropdown logic is the same as in the desktop copy.

**`PREP_TYPE`** (first setting below) chooses which analysis the output is for:

| PREP_TYPE | For | Certification columns added |
|---|---|---|
| `1` (default) | AASHE STARS 3.0 (OP-7) | AASHE Organic, AASHE Animal Welfare, AASHE Environment, AASHE Labor, AASHE Small Producers, AASHE Seafood |
| `2` | Anchors in Action alignment (GFPP, Real Food Challenge, Health Care Without Harm) | AW food group, AW, Env (with GFPP levels), No routine antibiotics, Labor (with GFPP levels), Local (GFPP level), Social Impact Supplier, and last, Program year and Nutrition checklist points (%) |

The other columns described below are added either way, except **Local Credit**, which is left out for `PREP_TYPE = 1` because STARS gives no credit for local purchases. Set `STARS_INCLUDE_LOCAL = True` to add it anyway.

Reads `input.csv` (an institutional food procurement spreadsheet). With `PREP_TYPE = 2` it appends three columns — **AW**, **Env** and **Labor** —
after the last used column, and gives every data cell in those columns a dropdown list of certifiers taken from the
**AIA** worksheet of `certifiers.ods`:

| New column | Options come from AIA rows where column A is |
|---|---|
| AW | `AW` (animal welfare, including GAP step levels) |
| Env | `Env` (environmental sustainability) |
| Labor | `Labor` |

Options are de-duplicated and sorted alphabetically, with **Other** added at the end of each list. Env and Labor options show each certification's GFPP Standards 3.0 level from the AIA sheet's **GFPP Level** column (e.g. `Level 3: USDA Organic`), grouped by level. AW levels depend on the food group (Standards 3.0 p. 34), so an **AW food group** column (Dairy, Eggs, Poultry, Beef, Pork, from the AIA sheet's **Food group** column) comes before AW, and the AW dropdown lists only that group's certifications with their levels (e.g. `Level 3: Global Animal Partnership: Step 4`). Each group's list is a named range on the Lists sheet (`AW_Beef` and so on) that the AW cell picks with `INDIRECT`.

For GFPP's Local and Community-Based Economies standard (Standards 3.0 pp. 18-19), `PREP_TYPE = 2` replaces the y-only Local and Social Impact columns with two dropdowns: **Local (GFPP level)**, whose three options give each level's supplier revenue cut-offs, and **Social Impact Supplier** (social/economic impact supplier, or POC supplier, which also counts as social/economic impact). The last two columns are dropdowns that hold one value for the whole file, chosen once in the first data row (row 2, highlighted): **Program year** (Year 1 to Year 25), because GFPP targets differ by year (Initial, Year 3, Year 5), and **Nutrition checklist points (%)** (0 to 100), the share of GFPP Community Health & Nutrition checklist points the institution earned (Standards 3.0 pp. 39-40).

For the Community Health & Nutrition antibiotics target (Standards 3.0 p. 38 and Appendix A), `PREP_TYPE = 2` adds **No routine antibiotics (y-cert or y)** right after Env. Each cell holds a formula that shows `y-cert` when the row's AW or Env choice is a certification marked `Yes` in the AIA sheet's **No routine antibiotics** column (its standard allows antibiotics only to treat or control disease, never for prevention or growth). Users may choose `y` instead (this replaces the formula) when the product has other third-party verification, such as Certified Responsible Antibiotic Use (CRAU) or a USDA Process Verified "No Antibiotics Ever" claim, and give the source in Explanation. The formula has no stored result until the file is saved in LibreOffice or Excel, so FoodImpacts should apply the same rule itself.

With `PREP_TYPE = 1` it appends instead six AASHE dropdowns, one per AASHE STARS 3.0 (OP-7) sustainable/ethical category: **AASHE Organic**, **AASHE Animal Welfare**, **AASHE Environment**, **AASHE Labor**, **AASHE Small Producers** and **AASHE Seafood**. Their options come from the **STARS** worksheet of `certifiers.ods` (built from AASHE's *STARS 3.0 Qualifying Food and Beverage Certifications and Programs*). Each column lists the certifications and programs in that section of the sheet, plus any others STARS marks as meeting that category, so a certification such as Regenerative Organic Certified appears in several columns. Campus farm or garden is in the Small Producers list. Users fill in every column that applies; STARS counts a purchase once toward its sustainable/ethical share however many categories it meets, and FoodImpacts does that counting. A visible **AASHE Lookup** sheet (PREP_TYPE 1 only) lists every option with the STARS categories it meets.

Four more columns follow: **Local Credit (enter y if yes)** and **Social Impact Supplier (enter y if yes)**, which accept only `y` or blank; **Plant Based (y or 1-4)**, which accepts `y`, a processing category `1`-`4` (NOVA groups, as AASHE STARS OP-7 uses) or blank; and free-text **Explanation (optional)**. The AASHE and Plant Based headings carry a cell note (comment) with a short guide, and a **Notes** sheet lists the Plant Based values with examples. With `SHOW_INPUT_MESSAGES = True` the guide also pops up whenever one of those cells is selected (off by default, because LibreOffice shows it as a yellow box that can cover the cells being filled in). Blank Plant Based cells are left for FoodImpacts to decide. The result is saved next to the input with "plus" added to its name (`input.csv` → `inputplus.xlsx`).

Requires: `pandas`, `openpyxl`, `odfpy` (`pip install pandas openpyxl odfpy`).

## Settings
Set `PREP_TYPE` first. Edit these paths if your files live elsewhere. Relative paths are relative to the folder the notebook is run from.

In [ ]:
# Parameters (papermill sets PREP_TYPE for each job):
#   1 = AASHE STARS 3.0 (OP-7), 2 = Anchors in Action alignment (GFPP)
PREP_TYPE = 1


In [ ]:
from pathlib import Path

# Which analysis the output is for (set to 1 or 2):
#   1 (default) = AASHE STARS 3.0 (OP-7): adds the six AASHE category dropdowns
#                 (AASHE_COLUMNS below) and AASHE Lookup sheet; leaves out AW, Env,
#                 Labor and (unless STARS_INCLUDE_LOCAL below) Local Credit.
#   2 = Anchors in Action alignment (GFPP, Real Food Challenge, Health Care Without
#       Harm): adds the AW, Env and Labor dropdowns; leaves out the AASHE dropdowns.
# PREP_TYPE is set in the cell above (the service sets it for each job).

# PREP_TYPE 1 only: set to True to add the Local Credit column anyway. STARS gives no
# credit for local purchases, so it adds nothing to the STARS score. PREP_TYPE 2 always
# includes it.
STARS_INCLUDE_LOCAL = True

# The AASHE and Plant Based guides are always a note on the column heading. Set to True to
# also show them whenever one of those cells is selected (LibreOffice shows this as a
# yellow box that can cover the cells being filled in).
SHOW_INPUT_MESSAGES = False

INPUT_CSV = Path("input.csv")
CERTIFIERS_ODS = Path("certifiers.ods")
CERTIFIERS_SHEET = "AIA"
# The service emails this file back
OUTPUT_XLSX = Path("output") / "worksheet_dropdowns.xlsx"
OUTPUT_XLSX.parent.mkdir(exist_ok=True)

# Category code in column A of the AIA sheet -> header of the new column
CATEGORIES = {"AW": "AW", "Env": "Env", "Labor": "Labor"}
# Categories whose options show the AIA sheet's GFPP Level, e.g. "Level 3: USDA Organic"
LEVELED_CATEGORIES = ["Env", "Labor"]
# AW levels depend on the food group (GFPP Standards 3.0 p. 34). PREP_TYPE 2 adds this column
# before AW, and the AW dropdown lists only the chosen group's certifications with their levels.
AW_FOOD_GROUP_HEADER = "AW food group"

# PREP_TYPE 2: GFPP Community Health & Nutrition antibiotics target (Standards 3.0 p. 38,
# Appendix A). This column goes right after Env. Its cells show "y-cert" (by formula) when the
# row's AW or Env certification is marked Yes in this column of the AIA sheet; users may choose
# "y" instead for other third-party verification.
ANTIBIOTICS_HEADER = "No routine antibiotics (y-cert or y)"
ANTIBIOTICS_AIA_COLUMN = "No routine antibiotics"
ANTIBIOTICS_CERT_VALUE = "y-cert"
ANTIBIOTICS_WIDTH = 16

# PREP_TYPE 2: GFPP Local and Community-Based Economies (Standards 3.0 pp. 18-19). These
# dropdowns replace the y-only Local and Social Impact columns.
GFPP_LOCAL_HEADER = "Local (GFPP level)"
GFPP_LOCAL_OPTIONS = [
    "Level 1: produce > $5M, other > $50M revenue",
    "Level 2: produce $1M-$5M, other $20M-$50M revenue",
    "Level 3: produce < $1M, other < $20M revenue",
]
GFPP_SOCIAL_HEADER = "Social Impact Supplier"
GFPP_SOCIAL_OPTIONS = [
    "Social/economic impact supplier",
    "POC supplier (also counts as social/economic impact)",
]
# Heading notes for the GFPP columns: header -> (title, text). Input messages allow 255 characters.
GFPP_GUIDES = {
    AW_FOOD_GROUP_HEADER: ("AW food group",
                           "Dairy (milk, yogurt, cheese), Eggs, Poultry (chicken, turkey, other "
                           "poultry), Beef (cattle and calves) or Pork. Choose it first: the AW "
                           "list shows that group's certifications."),
    "AW": ("AW (GFPP)",
           "GFPP Animal Welfare level for the chosen food group (Standards 3.0 p. 34). Level 3 "
           "meets the target at the lowest share. Choose the AW food group first."),
    "Env": ("Env (GFPP)",
            "GFPP Environmental Sustainability level (Standards 3.0 p. 24). Level 3 meets the "
            "target at the lowest share. Some seafood certifications count only for the "
            "products named in parentheses."),
    "Labor": ("Labor (GFPP)",
              "GFPP Valued Workforce level (Standards 3.0 p. 28), for farms and for processing, "
              "manufacturing and distribution. Fair Trade International counts only for coffee, "
              "sugar and cocoa. Union contract: e.g. UFW, UFCW, Teamsters, UNITE HERE."),
    ANTIBIOTICS_HEADER: ("No routine antibiotics (GFPP)",
                         "y-cert = the AW or Env certification bans routine antibiotic use "
                         "(automatic). Choose y only with other third-party verification, e.g. "
                         "CRAU or USDA Process Verified, and name it in Explanation. Animal "
                         "products only."),
    GFPP_LOCAL_HEADER: ("Local (GFPP)",
                        "Pick the level by the supplier's annual revenue. Every level also needs "
                        "private, cooperative, nonprofit or B Corp ownership, and production within "
                        "250 miles (500 for meat, poultry and seafood) or an alternative geographic radius approved by key "
                        "stakeholders."),
    GFPP_SOCIAL_HEADER: ("Social Impact (GFPP)",
                         "Counts only for local qualifying food. Social/economic impact includes "
                         "poverty or food insecurity, women, veterans, persons with disabilities and "
                         "People of Color. A POC supplier counts toward both GFPP targets."),
}

# PREP_TYPE 2: the last two columns hold one value for the whole file, chosen in the first
# data row (row 2). Program year: FoodImpacts compares results with that year's GFPP targets.
# FoodImpacts compares results with that year's GFPP targets (Initial, Year 3, Year 5).
PROGRAM_YEAR_LABEL = "Program year"
PROGRAM_YEARS = [f"Year {n}" for n in range(1, 26)]
# PREP_TYPE 2: percent of GFPP Community Health & Nutrition checklist points earned
# (Standards 3.0 pp. 39-40), a 0-100 dropdown
NUTRITION_POINTS_LABEL = "Nutrition checklist points (%)"
FILE_VALUE_GUIDES = {
    PROGRAM_YEAR_LABEL: ("Program year (GFPP)",
                         "Choose once, in the first row: the GFPP program year this purchase file "
                         "covers. FoodImpacts compares the results with that year's targets "
                         "(Initial, Year 3, Year 5)."),
    NUTRITION_POINTS_LABEL: ("Nutrition checklist (GFPP)",
                             "Choose once, in the first row: percent of GFPP Community Health & "
                             "Nutrition checklist points earned (pp. 39-40; items 1-11 = 2 points, "
                             "12-20 = 1 point; 33 in all). Targets: 51% initially, 80% by Year 5."),
}
FILE_VALUE_WIDTH = 13

# Added after the alphabetical certifiers at the end of every dropdown
OTHER_OPTION = "Other"

# Dropdowns added for PREP_TYPE 1, one per AASHE STARS 3.0 (OP-7) category, from this
# sheet of certifiers.ods. Each lists the rows in its Section plus the rows marked "Yes"
# in its category column.
STARS_SHEET = "STARS"
AASHE_COLUMNS = {  # header: (STARS Section, STARS category column)
    "AASHE Organic": ("Organic", "Organic"),
    "AASHE Animal Welfare": ("Animal welfare", "Animal welfare"),
    "AASHE Environment": ("Environment", "Environment"),
    "AASHE Labor": ("Labor", "Labor"),
    "AASHE Small Producers": ("Small producers", "Small producers"),
    "AASHE Seafood": ("Seafood", "Sustainable seafood"),
}
# The STARS sheet gives campus farms no section; they go in this column's list
CAMPUS_FARM_OPTION = "Campus farm or garden"
CAMPUS_FARM_COLUMN = "AASHE Small Producers"
# Guide for the AASHE columns (an input message allows 255 characters)
AASHE_PROMPT = ("Choose the product's certification or program in this STARS category. "
                "Fill in every AASHE column that applies; FoodImpacts counts a product "
                "once toward the STARS share. Categories and notes: AASHE Lookup sheet.")

# Columns added after the dropdown columns. FoodImpacts finds these, AW food group, AW, Env,
# ANTIBIOTICS_HEADER, Labor, PROGRAM_YEAR_LABEL, NUTRITION_POINTS_LABEL, the
# GFPP Local and Social headers and the AASHE_COLUMNS by their exact headings, so change
# them there too if you change them here.
EXTRA_COLUMNS = [
    "Local Credit (enter y if yes)",
    "Social Impact Supplier (enter y if yes)",
    "Plant Based (y or 1-4)",
    "Explanation (optional)",
]

# Column widths in the output workbook (headers wrap to fit). Dropdown columns are as wide as
# their longest option, up to MAX_DROPDOWN_WIDTH (AASHE columns are not capped, so their
# lists show whole names).
MAX_DROPDOWN_WIDTH = 40
Y_COLUMN_WIDTH = 13
TEXT_COLUMN_WIDTH = 30

# Of the columns above, these accept only "y" or blank (PREP_TYPE 2 replaces them with the
# GFPP Local and Social dropdowns)
Y_ONLY_COLUMNS = [
    "Local Credit (enter y if yes)",
    "Social Impact Supplier (enter y if yes)",
]

# The Plant Based column accepts y or a processing category 1-4 (NOVA groups, as
# AASHE STARS OP-7 uses). FoodImpacts needs this exact heading.
PLANT_BASED_HEADER = "Plant Based (y or 1-4)"
PLANT_BASED_VALUES = {
    "y": "Plant-based (counts under every definition)",
    "1": "Whole or minimally processed plant food (produce, plain beans, grains, nuts, "
         "single spices, tea, coffee)",
    "2": "Plant cooking ingredient (oils, sugars, syrups, starches, vinegar)",
    "3": "Simple processed plant food or meat/dairy alternative (100% juice, pasta, "
         "plain bread, tofu, hummus, nut butter, plant milk, vegan meat or cheese)",
    "4": "Ultra-processed plant food (most breakfast cereals, chips, snack bars, sweets, "
         "condiments, prepared frozen foods)",
    "blank": "Let FoodImpacts decide",
}
# Guide for the Plant Based column (an input message allows 255 characters; the
# examples are on the Notes sheet)
PLANT_BASED_PROMPT = ("y = plant-based\n"
                      "1 = whole or minimally processed\n"
                      "2 = plant cooking ingredient\n"
                      "3 = simple processed, or meat/dairy alternative\n"
                      "4 = ultra-processed\n"
                      "blank = FoodImpacts decides\n"
                      "Examples: see the Notes sheet")

## 1. Build the certifier lists from the AIA worksheet

In [ ]:
import re
import pandas as pd

if PREP_TYPE not in (1, 2):
    raise ValueError(f"PREP_TYPE must be 1 (AASHE STARS) or 2 (Anchors in Action), not {PREP_TYPE!r}")

aia = pd.read_excel(CERTIFIERS_ODS, sheet_name=CERTIFIERS_SHEET, engine="odf", header=None,
                    usecols=[0, 1, 2, 3, 4],
                    names=["category", "certifier", "food_group", "level", "no_antibiotics"],
                    dtype=str)
if str(aia.iloc[0, 4]).strip() != ANTIBIOTICS_AIA_COLUMN:
    raise ValueError(f"Column E of the {CERTIFIERS_SHEET} sheet should be headed {ANTIBIOTICS_AIA_COLUMN!r}")


def clean_name(name):
    name = str(name).strip()
    name = name.lstrip("•").strip()              # leading bullet
    name = re.sub(r"\s*\*+\s*$", "", name)       # trailing footnote asterisks
    name = re.sub(r"\s+", " ", name)              # collapse repeated spaces
    return name


aia = aia.dropna(subset=["category", "certifier"])
aia["category"] = aia["category"].astype(str).str.strip()
aia["certifier"] = aia["certifier"].map(clean_name)
aia["food_group"] = aia["food_group"].fillna("").str.strip()
aia["level"] = aia["level"].fillna("").str.strip().str.removesuffix(".0")
aia["no_antibiotics"] = aia["no_antibiotics"].fillna("").str.strip().str.casefold() == "yes"


def level_labels(rows):
    """'Level N: certifier' for each row, sorted by level then name."""
    rows = rows.sort_values(["level", "certifier"], key=lambda s: s.str.casefold())
    return [f"Level {lv}: {c}" for lv, c in zip(rows["level"], rows["certifier"])]


aw_by_group = {}  # PREP_TYPE 2: food group -> AW options for that group
if PREP_TYPE == 2:
    options = {}
    for code, header in CATEGORIES.items():
        if code == "AW":
            aw = aia[aia["category"] == code].drop_duplicates(["food_group", "certifier"])
            missing = aw.loc[(aw["food_group"] == "") | (aw["level"] == ""), "certifier"].tolist()
            if missing:
                raise ValueError(f"AW rows with no Food group or GFPP Level in the AIA sheet: {missing}")
            for group, rows in aw.groupby("food_group", sort=False):
                if not re.fullmatch(r"\w+", group):
                    raise ValueError(f"AW food group {group!r} must be one word (it names a range)")
                aw_by_group[group] = level_labels(rows) + [OTHER_OPTION]
            options[AW_FOOD_GROUP_HEADER] = list(aw_by_group)
            # Every group's options, used for the column width; the cell's list is its group's
            options[header] = sorted({n for names in aw_by_group.values() for n in names},
                                     key=str.casefold)
            continue
        rows = aia[aia["category"] == code].drop_duplicates("certifier")
        if code in LEVELED_CATEGORIES:
            missing = rows.loc[rows["level"] == "", "certifier"].tolist()
            if missing:
                raise ValueError(f"{code} rows with no GFPP Level in the AIA sheet: {missing}")
            names = level_labels(rows)
        else:
            names = sorted(rows["certifier"], key=str.casefold)
        options[header] = names + [OTHER_OPTION]
    # AW and Env options whose certification is marked Yes for no routine antibiotics
    antibiotic_options = []
    for code in ("AW", "Env"):
        flagged = set(aia.loc[(aia["category"] == code) & aia["no_antibiotics"], "certifier"])
        antibiotic_options += [n for n in options[CATEGORIES[code]]
                               if n.split(": ", 1)[-1] in flagged]
    options[GFPP_LOCAL_HEADER] = GFPP_LOCAL_OPTIONS
    options[GFPP_SOCIAL_HEADER] = GFPP_SOCIAL_OPTIONS
else:
    stars = pd.read_excel(CERTIFIERS_ODS, sheet_name=STARS_SHEET, engine="odf", dtype=str).fillna("")
    # Each column names its category, so drop the section prefix (e.g. "Organic: ")
    stars["Dropdown option"] = [opt.removeprefix(f"{sec}: ") for sec, opt
                                in zip(stars["Section"], stars["Dropdown option"])]
    stars = stars.sort_values(["Section", "Dropdown option"], key=lambda s: s.str.casefold())
    options = {}
    for header, (section, category) in AASHE_COLUMNS.items():
        rows = stars[(stars["Section"] == section) | (stars[category] == "Yes")]
        names = list(rows["Dropdown option"])
        if header == CAMPUS_FARM_COLUMN:
            names.append(CAMPUS_FARM_OPTION)
        options[header] = sorted(names, key=str.casefold) + [OTHER_OPTION]

for header, names in {**options, **{f"AW ({g})": n for g, n in aw_by_group.items()}}.items():
    if header == "AW" and aw_by_group:
        continue  # shown per food group below
    print(f"{header} ({len(names)} options)")
    for n in names:
        print("   ", n)
if PREP_TYPE == 2:
    print(f"{ANTIBIOTICS_CERT_VALUE} when AW or Env is one of these ({len(antibiotic_options)}):")
    for n in antibiotic_options:
        print("   ", n)


## 2. Read input.csv and add the new columns

In [ ]:
try:
    data = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
except UnicodeDecodeError:
    data = pd.read_csv(INPUT_CSV, encoding="cp1252")

# Drop trailing columns that are completely empty (e.g. stray commas at line ends),
# so the new columns start at the first empty column after the data.
def is_empty(col):
    header_blank = str(col).startswith("Unnamed:") or not str(col).strip()
    return header_blank and data[col].isna().all()

while len(data.columns) and is_empty(data.columns[-1]):
    data = data.drop(columns=data.columns[-1])

# STARS gives no credit for local purchases, so PREP_TYPE 1 leaves out the Local column
# unless STARS_INCLUDE_LOCAL is True
extra_columns = [h for h in EXTRA_COLUMNS
                 if not (PREP_TYPE == 1 and not STARS_INCLUDE_LOCAL
                         and h.startswith("Local Credit"))]
# PREP_TYPE 2 has the GFPP Local and Social dropdowns in place of the y-only columns
if PREP_TYPE == 2:
    extra_columns = [h for h in extra_columns if h not in Y_ONLY_COLUMNS]

for header in [*options, *extra_columns]:
    if header in data.columns:
        raise ValueError(f"input.csv already has a column named {header!r}")
    data[header] = None

# PREP_TYPE 2: the antibiotics column goes right after Env
if PREP_TYPE == 2:
    if ANTIBIOTICS_HEADER in data.columns:
        raise ValueError(f"input.csv already has a column named {ANTIBIOTICS_HEADER!r}")
    data.insert(data.columns.get_loc(CATEGORIES["Env"]) + 1, ANTIBIOTICS_HEADER, None)
    # Whole-file values, last
    for header in FILE_VALUE_GUIDES:
        if header in data.columns:
            raise ValueError(f"input.csv already has a column named {header!r}")
        data[header] = None

print(f"{len(data)} rows, {len(data.columns)} columns")
data.tail()

## 3. Write <input name>plus.xlsx with dropdowns

The option lists are stored on a hidden `Lists` sheet and the dropdowns reference those ranges.
(Excel limits a typed-in dropdown list to 255 characters, which the AW list exceeds.)

In [ ]:
import math
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.worksheet.datavalidation import DataValidation

def add_guide(dv, col, title, text):
    """Put a column's guide in a note on its heading, and optionally in its input message."""
    ws[f"{col}1"].comment = Comment(f"{title}\n{text}", "add_certifier_dropdowns",
                                    width=320, height=170)
    if SHOW_INPUT_MESSAGES:
        dv.showInputMessage = True
        dv.promptTitle = title
        dv.prompt = text


with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    data.to_excel(writer, sheet_name="Data", index=False)
    wb = writer.book
    ws = writer.sheets["Data"]

    lists_ws = wb.create_sheet("Lists")
    lists_ws.sheet_state = "hidden"

    def add_list(title, names):
        """Write names to the next column of the Lists sheet and return their range."""
        col = lists_ws.max_column + 1 if lists_ws.max_row > 1 or lists_ws["A1"].value else 1
        lists_ws.cell(row=1, column=col, value=title)
        for r, name in enumerate(names, start=2):
            lists_ws.cell(row=r, column=col, value=name)
        letter = get_column_letter(col)
        return f"Lists!${letter}$2:${letter}${len(names) + 1}"

    last_row = max(len(data) + 1, 2)  # header is row 1
    for header, names in options.items():
        data_col = get_column_letter(data.columns.get_loc(header) + 1)
        if header == "AW" and aw_by_group:
            # One named list per food group (AW_Beef, ...); each AW cell shows the list
            # named by the food group in its row
            for group, group_names in aw_by_group.items():
                name = f"AW_{group}"
                wb.defined_names.add(DefinedName(name, attr_text=add_list(f"AW {group}", group_names)))
            group_col = get_column_letter(data.columns.get_loc(AW_FOOD_GROUP_HEADER) + 1)
            source = f'INDIRECT("AW_"&${group_col}2)'
            error = "Choose the AW food group first, then a certification from its list."
        else:
            source = "=" + add_list(header, names)
            error = f"Please choose an option from the {header} list."
        dv = DataValidation(type="list", formula1=source, allow_blank=True,
                            showErrorMessage=True, errorTitle="Invalid entry", error=error)
        if header in AASHE_COLUMNS:
            add_guide(dv, data_col, "AASHE STARS", AASHE_PROMPT)
        elif PREP_TYPE == 2 and header in GFPP_GUIDES:
            add_guide(dv, data_col, *GFPP_GUIDES[header])
        dv.add(f"{data_col}2:{data_col}{last_row}")
        ws.add_data_validation(dv)

        # These lists are short, so show their options in full
        uncapped = header in AASHE_COLUMNS or header in ("Labor", GFPP_LOCAL_HEADER, GFPP_SOCIAL_HEADER)
        width = max(len(n) for n in names) + 2
        ws.column_dimensions[data_col].width = width if uncapped else min(width, MAX_DROPDOWN_WIDTH)

    if PREP_TYPE == 2:
        # y-cert by formula when the row's AW or Env choice is on the antibiotics list.
        # SUMPRODUCT compares exactly (COUNTIF/MATCH would treat * ? ~ as wildcards).
        wb.defined_names.add(DefinedName("NoRoutineAntibiotics", attr_text=add_list(
            ANTIBIOTICS_AIA_COLUMN, antibiotic_options)))
        abx_col = get_column_letter(data.columns.get_loc(ANTIBIOTICS_HEADER) + 1)
        aw_col = get_column_letter(data.columns.get_loc(CATEGORIES["AW"]) + 1)
        env_col = get_column_letter(data.columns.get_loc(CATEGORIES["Env"]) + 1)
        for r in range(2, len(data) + 2):
            ws[f"{abx_col}{r}"] = (f"=IF(SUMPRODUCT((NoRoutineAntibiotics=${aw_col}{r})"
                                   f"+(NoRoutineAntibiotics=${env_col}{r}))>0,"
                                   f'"{ANTIBIOTICS_CERT_VALUE}","")')
        dv = DataValidation(type="list", formula1='"y"', allow_blank=True,
                            showErrorMessage=True, errorTitle="Invalid entry",
                            error=f"Choose y (other third-party verification), or keep the "
                                  f"{ANTIBIOTICS_CERT_VALUE} formula.")
        add_guide(dv, abx_col, *GFPP_GUIDES[ANTIBIOTICS_HEADER])
        dv.add(f"{abx_col}2:{abx_col}{last_row}")
        ws.add_data_validation(dv)
        ws.column_dimensions[abx_col].width = ANTIBIOTICS_WIDTH

    if PREP_TYPE == 2:
        # Whole-file values: a dropdown in the first data row only, highlighted
        file_value_lists = {PROGRAM_YEAR_LABEL: f'"{",".join(PROGRAM_YEARS)}"',
                            NUTRITION_POINTS_LABEL: "=" + add_list(NUTRITION_POINTS_LABEL,
                                                                     list(range(101)))}
        for header, source in file_value_lists.items():
            col = get_column_letter(data.columns.get_loc(header) + 1)
            dv = DataValidation(type="list", formula1=source, allow_blank=True,
                                showErrorMessage=True, errorTitle="Invalid entry",
                                error=f"Please choose an option from the {header} list.")
            add_guide(dv, col, *FILE_VALUE_GUIDES[header])
            dv.add(f"{col}2")
            ws.add_data_validation(dv)
            ws[f"{col}2"].fill = PatternFill("solid", fgColor="FFF2CC")
            ws.column_dimensions[col].width = FILE_VALUE_WIDTH

    for header in extra_columns:
        data_col = get_column_letter(data.columns.get_loc(header) + 1)
        if header in Y_ONLY_COLUMNS:
            ws.column_dimensions[data_col].width = Y_COLUMN_WIDTH
            dv = DataValidation(type="list", formula1='"y"', allow_blank=True,
                                showErrorMessage=True, errorTitle="Invalid entry",
                                error="Enter y for yes, or leave the cell blank.")
            dv.add(f"{data_col}2:{data_col}{last_row}")
            ws.add_data_validation(dv)
        elif header == PLANT_BASED_HEADER:
            ws.column_dimensions[data_col].width = Y_COLUMN_WIDTH
            values = [v for v in PLANT_BASED_VALUES if v != "blank"]
            dv = DataValidation(type="list", formula1=f'"{",".join(values)}"', allow_blank=True,
                                showErrorMessage=True, errorTitle="Invalid entry",
                                error="Enter y, a category 1-4, or leave blank.")
            add_guide(dv, data_col, "Plant Based", PLANT_BASED_PROMPT)
            dv.add(f"{data_col}2:{data_col}{last_row}")
            ws.add_data_validation(dv)
        else:
            ws.column_dimensions[data_col].width = TEXT_COLUMN_WIDTH

    # Wrap the header row and make it tall enough for the longest wrapped header
    lines = 1
    for cell in ws[1]:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
        width = ws.column_dimensions[cell.column_letter].width or 8.43
        lines = max(lines, math.ceil(len(str(cell.value)) / max(width - 2, 1)))
    ws.row_dimensions[1].height = 15 * lines

    ws.freeze_panes = "A2"

    notes_ws = wb.create_sheet("Notes", index=1)
    if PREP_TYPE == 2:
        notes_ws.append(["Program year column", "Last column but one; choose once, in the first data "
                         "row. The GFPP program year this purchase file covers. "
                         "FoodImpacts compares the results with that year's GFPP targets "
                         "(Initial, Year 3, Year 5)."])
        notes_ws.append(["Nutrition points column", "Last column; choose once, in the first data row. "
                         "The percent of GFPP Community Health & "
                         "Nutrition checklist points earned (Standards 3.0 pp. 39-40; high priority "
                         "items 1-11 are worth 2 points, priority items 12-20 are worth 1). "
                         "Targets: 51% initially, 80% by Year 5."])
        notes_ws.append(["No routine antibiotics", "Animal products raised without antibiotics "
                         "for disease prevention or other routine purposes (Standards 3.0 p. 38; "
                         "Appendix A). Treatment, and control of a disease shown to be present, "
                         "are allowed. Targets: chicken 40% initially, 75% by Year 5; all other "
                         f"animal products 15%, then 30%. {ANTIBIOTICS_CERT_VALUE} appears by formula "
                         "when the AW or Env certification qualifies (list: hidden Lists sheet, from "
                         "the AIA sheet of certifiers.ods). It may also appear on plant foods (e.g. "
                         "organic carrots); FoodImpacts counts animal products only. Choose y only "
                         "with other third-party verification and name it in Explanation: Certified "
                         "Responsible Antibiotic Use (CRAU), or a USDA 'No Antibiotics Ever' or "
                         "'Raised Without Antibiotics' claim under a USDA Process Verified Program "
                         "or another certifier (the label alone rests on the producer's word). "
                         "Choosing y replaces the formula; to restore it, copy a cell from another row."])
        notes_ws.append([])

    # Visible key to the Plant Based values
    notes_ws.append([PLANT_BASED_HEADER, "Meaning"])
    for value, meaning in PLANT_BASED_VALUES.items():
        notes_ws.append([value, meaning])
    notes_ws.column_dimensions["A"].width = 22
    notes_ws.column_dimensions["B"].width = 110

    if PREP_TYPE == 2:
        notes_ws.append([])
        for header, (title, text) in GFPP_GUIDES.items():
            notes_ws.append([header, text])
        for row in notes_ws.iter_rows(min_col=2, max_col=2):
            row[0].alignment = Alignment(wrap_text=True, vertical="top")

    if PREP_TYPE == 1:
        notes_ws.append([])
        notes_ws.append(["AASHE columns", "Each AASHE column is one STARS sustainable/ethical category. "
                         "Fill in every column that applies. STARS counts a purchase that meets more "
                         "than one category only once toward its sustainable/ethical share, and "
                         "FoodImpacts does that counting. A product that is also plant-based counts "
                         "in both STARS figures."])
        notes_ws.append(["", "The AASHE Lookup sheet shows the STARS categories each option meets."])

        # Visible list of the AASHE options and the STARS categories each meets
        stars.to_excel(writer, sheet_name="AASHE Lookup", index=False)
        lookup_ws = writer.sheets["AASHE Lookup"]
        wb.move_sheet(lookup_ws, offset=wb.index(notes_ws) + 1 - wb.index(lookup_ws))
        for col, width in zip("ABCDEFGHIJKLM", [16, 50, 40, 30, 10, 11, 10, 10, 12, 11, 12, 35, 60]):
            lookup_ws.column_dimensions[col].width = width
        for cell in lookup_ws[1]:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        lookup_ws.freeze_panes = "C2"

print(f"Saved {OUTPUT_XLSX.resolve()}")